# AML Transaction Monitoring & Alert Investigation Analytics
This notebook analyzes synthetic customer and transaction data for AML red flags, behavioral patterns, and customer risk scoring.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('..')
customers = pd.read_csv(ROOT/'data/raw/customers.csv')
tx = pd.read_csv(ROOT/'data/processed/transactions_enriched.csv', parse_dates=['transaction_date'])
alerts = pd.read_csv(ROOT/'data/processed/alerts.csv')
risk = pd.read_csv(ROOT/'data/processed/customer_risk_features.csv')

customers.shape, tx.shape, alerts.shape, risk.shape

## 1. Transaction Overview

In [ ]:
tx[['amount']].describe()


## 2. Channel Mix

In [ ]:
tx['channel'].value_counts(normalize=True).mul(100).round(1)

## 3. High-Risk Jurisdiction Activity

In [ ]:
tx[tx['counterparty_country'].isin(['RU','IR','SY'])]\
  .groupby('customer_id')['amount'].agg(['count','sum']).sort_values('sum', ascending=False).head(20)

## 4. Structuring-Like Cash Deposits

In [ ]:
structuring = tx[(tx['channel']=='Cash Deposit') & tx['amount'].between(8000,9999.99)]
structuring.groupby('customer_id')['amount'].agg(['count','sum']).query('count >= 3').sort_values('sum', ascending=False)

## 5. Customer AML Risk Scores

In [ ]:
risk.sort_values('risk_score', ascending=False).head(25)

## 6. Alert Analysis

In [ ]:
alerts.groupby('scenario').agg(alerts=('alert_id','count'), avg_risk=('risk_score','mean'), alert_volume=('alert_amount','sum')).round(2)